<a href="https://colab.research.google.com/github/asoknaren/AIPlayground/blob/main/001_tokenizer_sample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import time

In [ ]:
MODEL_NAME = "microsoft/phi-3-mini-4k-instruct"
TARGET_WORDS = 200

In [ ]:
def build_prompt() -> str:
    return (
        "Write a sincere professional apology email in about 200 words. "
        "Context: I missed an important client deadline due to poor planning. "
        "The email should acknowledge responsibility, explain briefly without excuses, "
        "and include a clear corrective action plan and request to rebuild trust."
    )


In [ ]:
prompt = build_prompt()

In [ ]:
def initialize_model_and_tokenizer(model_name: str):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    if device == "cpu":
        cpu_threads = max(1, (torch.get_num_threads() or 1) - 1)
        torch.set_num_threads(cpu_threads)
        print(f"Using CPU threads: {cpu_threads}")

    return tokenizer, model, device

In [ ]:
tokenizer, model, device = initialize_model_and_tokenizer(MODEL_NAME)

In [ ]:
def tokenize_and_display_prompt_info(tokenizer, prompt: str, device: str):
    messages = [{"role": "user", "content": prompt}]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    model_inputs = {k: v.to(device) for k, v in model_inputs.items()}
    token_count = model_inputs["input_ids"].shape[1]

    print(f"Prompt token count: {token_count}")
    print(f"Equivalent tokenized words: {tokenizer.decode(model_inputs['input_ids'][0])}")

    print("\nToken to Word Mapping:")
    for token_id in model_inputs['input_ids'][0]:
        token_word = tokenizer.decode(token_id)
        print(f"Token ID: {token_id.item()}, Word: '{token_word}'")

    return model_inputs, token_count

In [ ]:
model_inputs, token_count = tokenize_and_display_prompt_info(tokenizer, prompt, device)

In [ ]:
def generate_output(tokenizer, model, prompt: str, target_words: int, device: str) -> None:

    max_new_tokens = 240 if device == "cpu" else 320
    start = time.perf_counter()
    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    elapsed_s = time.perf_counter() - start

    new_tokens = generated_ids[0, token_count:]
    output_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    generated_tokens = new_tokens.shape[0]
    tps = generated_tokens / elapsed_s if elapsed_s > 0 else 0.0
    print(
        f"Generated tokens: {generated_tokens} in {elapsed_s:.1f}s "
        f"({tps:.2f} tokens/sec, target words: {target_words})"
    )
    print("\nGenerated output:\n")
    print(output_text)




In [ ]:
generate_output(tokenizer, model, prompt, TARGET_WORDS, device)